# 03 — Summarization Compression — Practical Examples

**Covers**: compress_history helper, RollingSummaryContext class, hierarchical compression, trigger rules, agent loop demo

In [ ]:
import os, json
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
enc = tiktoken.encoding_for_model(MODEL)

def count_tokens(messages: list[dict]) -> int:
    return sum(3 + len(enc.encode(str(m.get('content','')))) for m in messages) + 3

print('✅ Setup complete')

## 1. Core Summarizer — Compress Messages to a Paragraph

In [ ]:
def compress_to_summary(messages: list[dict], existing_summary: str = None) -> str:
    """Use LLM to compress messages into a concise summary."""
    text = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages if m['role'] != 'system'
    )
    if existing_summary:
        prompt = f"""Update this summary with new conversation. Be concise (max 3 paragraphs).

Current summary:
{existing_summary}

New conversation:
{text}"""
    else:
        prompt = f"Summarize this conversation concisely (max 2 paragraphs):\n\n{text}"
    
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300, temperature=0.0
    )
    return r.choices[0].message.content

# Build a realistic 10-turn conversation
conversation = [
    {'role': 'user',      'content': 'I need to build a REST API for a todo app. Where should I start?'},
    {'role': 'assistant', 'content': 'Start with FastAPI. Install it with pip install fastapi uvicorn. Create main.py with a FastAPI app instance.'},
    {'role': 'user',      'content': 'Great. How do I define a Todo model?'},
    {'role': 'assistant', 'content': 'Use Pydantic BaseModel: class Todo(BaseModel): id: int; title: str; done: bool = False'},
    {'role': 'user',      'content': 'How do I add a POST endpoint to create todos?'},
    {'role': 'assistant', 'content': '@app.post("/todos") async def create_todo(todo: Todo): todos.append(todo); return todo'},
    {'role': 'user',      'content': 'How do I add a database instead of in-memory list?'},
    {'role': 'assistant', 'content': 'Use SQLAlchemy with SQLite. Create a database.py with engine, SessionLocal, and Base. Use dependency injection for sessions.'},
]

original_tokens = count_tokens(conversation)
print(f"Original: {len(conversation)} messages, {original_tokens} tokens")

summary = compress_to_summary(conversation)
summary_tokens = len(enc.encode(summary))
print(f"Summary:  {summary_tokens} tokens ({summary_tokens/original_tokens:.0%} of original)")
print(f"\n=== Summary ===\n{summary}")

## 2. RollingSummaryContext — Auto-Compresses as History Grows

In [ ]:
class RollingSummaryContext:
    def __init__(self, system: str, compress_at_tokens: int = 600, model: str = MODEL):
        self.system = system
        self.compress_at = compress_at_tokens
        self.model = model
        self.enc = tiktoken.encoding_for_model(model)
        self._history: list[dict] = []
        self._summary: str | None = None
        self._n_compressions = 0

    def add(self, role: str, content: str):
        self._history.append({'role': role, 'content': content})
        if self._history_tokens() > self.compress_at:
            self._compress()

    def _history_tokens(self) -> int:
        return sum(3 + len(self.enc.encode(str(m.get('content','')))) for m in self._history)

    def _compress(self):
        half = len(self._history) // 2
        to_compress = self._history[:half]
        self._history = self._history[half:]
        self._summary = compress_to_summary(to_compress, self._summary)
        self._n_compressions += 1
        print(f"  🗜️  Compression #{self._n_compressions}: {len(to_compress)} messages → summary")

    def get_messages(self) -> list[dict]:
        msgs = [{'role': 'system', 'content': self.system}]
        if self._summary:
            msgs.append({'role': 'assistant', 'content': f'[Conversation summary]\n{self._summary}'})
        return msgs + self._history

    def stats(self) -> dict:
        msgs = self.get_messages()
        total = sum(3 + len(self.enc.encode(str(m.get('content','')))) for m in msgs) + 3
        return {'window_messages': len(msgs), 'window_tokens': total,
                'history_len': len(self._history), 'compressions': self._n_compressions,
                'has_summary': self._summary is not None}

# Test with a multi-turn coding agent session
ctx = RollingSummaryContext(
    system="You are a Python coding tutor. Help the student learn step by step.",
    compress_at_tokens=400   # Low threshold for demo
)

topics = [
    "Explain Python variables",
    "Now explain data types",
    "What are lists?",
    "How do I loop over a list?",
    "What are dictionaries?",
]

for topic in topics:
    ctx.add('user', topic)
    r = client.chat.completions.create(model=MODEL, messages=ctx.get_messages(), max_tokens=80)
    answer = r.choices[0].message.content
    ctx.add('assistant', answer)
    s = ctx.stats()
    print(f"  After '{topic[:30]}': {s['window_tokens']} tokens, summary={s['has_summary']}")

print(f"\nFinal: {ctx.stats()}")

## 3. Compare: Sliding Window vs Summarization

In [ ]:
# Simulate 20 turns and compare what each strategy retains
def sliding_window_tokens(messages, max_tokens=800):
    if not messages: return messages
    system = [messages[0]] if messages[0]['role'] == 'system' else []
    history = messages[len(system):]
    kept, used = [], sum(3 + len(enc.encode(str(m.get('content','')))) for m in system) + 3
    for m in reversed(history):
        t = 3 + len(enc.encode(str(m.get('content',''))))
        if used + t <= max_tokens: kept.append(m); used += t
        else: break
    return system + list(reversed(kept))

# Build 20 turn history
history = [{'role': 'system', 'content': 'You are a Python tutor.'}]
facts = [
    ('variables', 'Variables store data. Use = to assign: x = 5'),
    ('lists',     'Lists are ordered: [1, 2, 3]. Access with index: lst[0]'),
    ('dicts',     'Dicts map keys to values: {"name": "Alice"}'),
    ('loops',     'for item in list: print(item)'),
    ('functions', 'def greet(name): return f"Hello {name}"'),
    ('classes',   'class Dog: def __init__(self, name): self.name = name'),
    ('errors',    'try: code except Exception as e: handle(e)'),
    ('files',     'with open("file.txt") as f: content = f.read()'),
]
for topic, fact in facts:
    history.append({'role': 'user',      'content': f'Tell me about {topic}'})
    history.append({'role': 'assistant', 'content': fact})

total = count_tokens(history)
slid  = sliding_window_tokens(history, 400)
slid_t = count_tokens(slid)

print(f"{'Strategy':<22} {'Messages':>10} {'Tokens':>10} {'Retains'}")
print('-' * 60)
print(f"{'Original':<22} {len(history):>10} {total:>10} all topics")
print(f"{'Sliding 400 tokens':<22} {len(slid):>10} {slid_t:>10} last ~{len(slid)-1} messages only")
print()
print("Topics visible in sliding window:")
for m in slid:
    if m['role'] == 'user' and 'system' not in m['role']:
        print(f"  ✅ {m['content']}")

## 4. Compression Trigger — When to Summarize

In [ ]:
def compression_needed(messages: list[dict], context_limit: int = 128_000, trigger_pct: float = 0.6) -> bool:
    current = count_tokens(messages)
    threshold = context_limit * trigger_pct
    return current >= threshold

# Model-specific trigger thresholds
COMPRESSION_CONFIG = {
    "gpt-4o-mini":     {"limit": 128_000, "trigger_at": 0.60},
    "gpt-4o":          {"limit": 128_000, "trigger_at": 0.65},
    "claude-3-haiku":  {"limit": 200_000, "trigger_at": 0.60},
    "gemini-1.5-flash":{"limit": 1_000_000, "trigger_at": 0.70},
}

# Demo the decision logic
print(f"Context limit: 128,000 tokens | Trigger: 60%")
for tokens in [10_000, 40_000, 76_800, 90_000, 120_000]:
    mock_msgs = [{'role': 'user', 'content': 'x' * (tokens * 3)}]  # rough approx
    # Use fixed token count for demo
    needed = tokens >= 128_000 * 0.60
    status = '🔴 COMPRESS' if needed else '🟢 OK'
    print(f"  At {tokens:>7,} tokens ({tokens/128000:.0%}): {status}")

## 📌 Summary

- **Summarization** preserves old context intelligently vs hard-dropping with sliding window
- **compress_to_summary()**: send old messages to LLM → get ~200 token summary
- **RollingSummaryContext**: auto-compresses when threshold hit, injects summary into context
- **Trigger at 60%**: compress early — way before hitting the hard limit
- **Trade-off**: summarization costs tokens to produce the summary (plan for it)
- **Combine strategies**: sliding window for raw recency + summarization for older history